<a href="https://colab.research.google.com/github/Chepngetich/AI-for-Science-and-Engineering/blob/main/DArTseq_Genotype_Imputation_ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI/ML Course Project: Machine-Learning Imputation of Missing DArTseq SNP Genotypes

## 1. Project problem and intended AI/ML outcome

**Problem:** Sequencing-derived SNP datasets such as DArTseq can contain missing or uncertain genotype calls because of factors such as sequencing depth, marker quality, allele frequency and technical variation. Missing genotypes can reduce the accuracy and statistical power of downstream analyses including GWAS, genomic prediction, genetic diversity assessment and population-structure analysis.

**AI/ML outcome:** I would like to investigate whether machine learning can accurately predict missing SNP genotype values from genomic patterns present in other SNP markers. The immediate outcome of this assignment is a supervised ML baseline that predicts deliberately masked genotype calls and compares the predictions with the original known genotypes.

**Project question:** *Can machine learning accurately recover missing DArTseq SNP genotypes from the observed genotypes of the same individuals?*


## 2. Data needed

The project needs:

- DArTseq SNP genotype calls for many individuals.
- Multiple polymorphic SNP markers so that genomic relationships can be learned.
- Missing genotype information and/or a way to create a controlled missingness experiment.
- SNP marker identifiers and, where available, marker quality information.
- Ideally, sequencing/read-depth information and reference-panel information for a future, more advanced model.

For this assignment I use a **real public DArTseq dataset** containing **8,649 SNPs from 430 koala individuals**. The dataset is described as an **unfiltered SNP dataset**, and the genotyping was performed by Diversity Arrays Technology using DArTseq.


## 3. Dataset used

**Dataset:** *Testing the effectiveness of genetic monitoring using genetic non-invasive sampling*

**Source:** Dryad / Zenodo

**DArTseq file:** `genotypes_file.csv` (~17.9 MB)

**Dryad:** https://doi.org/10.5061/dryad.1ns1rn8vq

**Zenodo record:** https://zenodo.org/records/7470232

The published repository reports 8,649 SNPs for 430 individuals and states that the SNP data were generated using DArTseq. The repository also provides `idmeta.csv`, `observedIR.csv`, `Tracking_data.csv`, and a README.

### Why this dataset is appropriate?

It is directly related to the proposed project because it is real **DArTseq SNP genotype data**, rather than a generic tabular ML dataset. The data can be used to benchmark genotype prediction by temporarily masking known genotype calls and asking an ML model to recover them.



In [ ]:
# 3A. Download/import the real DArTseq genotype file
#
# The dataset is publicly available from Dryad. The repository lists
# genotypes_file.csv (~17.9 MB). If automatic download is unavailable,
# download the file from the Dryad page and use the upload cell below.

import requests
from pathlib import Path

FILE = Path("genotypes_file.csv")
DRYAD_FILE_URL = "https://datadryad.org/downloads/file_stream/1216603"

if not FILE.exists():
    try:
        r = requests.get(DRYAD_FILE_URL, timeout=180, allow_redirects=True)
        r.raise_for_status()
        FILE.write_bytes(r.content)
        print("Downloaded genotypes_file.csv successfully.")
    except Exception as e:
        print("Automatic download was not available:", e)
        print("Please use the upload cell below.")

if FILE.exists():
    print(f"File: {FILE}")
    print(f"Size: {FILE.stat().st_size/1e6:.2f} MB")


In [ ]:
# 3B. Manual upload fallback
# Run this cell if the automatic download above was unsuccessful.

from pathlib import Path
from google.colab import files

FILE = Path("genotypes_file.csv")

if not FILE.exists():
    uploaded = files.upload()
    if "genotypes_file.csv" not in uploaded:
        raise ValueError("Please upload the file named genotypes_file.csv from the Dryad dataset.")
    print("Uploaded genotypes_file.csv")


In [ ]:
# 4. Import the dataset and display a small portion

import pandas as pd
import numpy as np

if not FILE.exists():
    raise FileNotFoundError("genotypes_file.csv is not available. Download it from Dryad or run the upload cell.")

raw = pd.read_csv(FILE, low_memory=False)

print("Raw dataset shape:", raw.shape)
display(raw.head(10))
print("\nFirst 20 column names:")
print(raw.columns[:20].tolist())


### Data inspection

The next cells inspect the structure before modelling. This is important because DArTseq files can contain marker metadata alongside genotype columns, and the exact genotype coding should be verified rather than assumed.


In [ ]:
# Inspect data types and missing-value symbols

print(raw.dtypes.value_counts())
print("\nPotential missing-value strings:")
for value in ["-", "", "NA", "N/A", "null"]:
    print(value, int((raw.astype(str) == value).sum().sum()))


In [ ]:
# Convert common missing-value symbols to NaN and inspect numeric-like columns

df = raw.replace(["-", "", "NA", "N/A", "null", "None"], np.nan)

numeric = df.apply(pd.to_numeric, errors="coerce")
numeric_fraction = numeric.notna().mean()

print("Columns with >=80% numeric values:", int((numeric_fraction >= 0.80).sum()))
print(numeric_fraction.sort_values(ascending=False).head(20))


In [ ]:
# Identify likely genotype columns.
# For this project we expect diploid genotype states coded as 0, 1 and 2.
# The check is deliberately explicit so we do not silently misinterpret DArTseq data.

geno_like = []
for col in numeric.columns:
    vals = set(numeric[col].dropna().unique())
    if len(vals) > 0 and vals.issubset({0, 1, 2}):
        geno_like.append(col)

print("Columns whose observed numeric values are only 0/1/2:", len(geno_like))
print(geno_like[:20])


### If no 0/1/2 genotype columns are detected

DArTseq datasets can be delivered in different report formats. If the previous cell reports zero or very few 0/1/2 columns, **do not proceed blindly**. Inspect `raw.head()` and the unique values and adapt the parsing rule to the actual genotype representation. The project should document that conversion explicitly.


In [ ]:
# Build an individual x SNP genotype matrix when 0/1/2 coding is detected.
# The real DArTseq file may be SNP x individual, so the orientation is checked.

if len(geno_like) < 20:
    raise ValueError("Fewer than 20 columns were detected as 0/1/2 genotype columns. Inspect the DArTseq file format before continuing.")

G0 = numeric[geno_like].copy()

# If there are substantially more rows than genotype columns, the file is likely SNP x individual.
# Transpose to obtain individuals x SNPs for modelling.
if G0.shape[0] > G0.shape[1] * 5:
    G = G0.T
    print("Orientation detected as approximately SNP x individual; matrix transposed.")
else:
    G = G0.copy()
    print("Orientation retained as approximately individual x SNP.")

print("Genotype matrix shape (individuals x SNPs):", G.shape)
display(G.iloc[:5, :10])


## 5. Potential inputs (features) and output (target)

### Potential input features

The main input features are the **observed genotypes at other SNP markers** for the same individual. These markers capture genomic patterns that may help predict the genotype at a target SNP.

For this first experiment, I will use a manageable number of informative SNPs rather than all 8,649 markers. This makes the baseline easier to understand and run in Google Colab.

### Target

The target is the genotype state at one selected SNP. For the verified 0/1/2 coding, the three classes represent the two homozygous states and the heterozygous state. The exact biological interpretation depends on the allele coding in the source data.

### Controlled missingness

To evaluate the model objectively, I will select genotype values that are originally observed, temporarily hide a subset of them, train/test the model, and compare predictions against the original values. This gives us a known ground truth.


In [ ]:
# Quantify missingness in the genotype matrix

missing_by_snp = G.isna().mean().sort_values(ascending=False)
missing_by_individual = G.isna().mean(axis=1).sort_values(ascending=False)

print("Overall genotype missingness:", round(G.isna().mean().mean()*100, 2), "%")
print("Median SNP missingness:", round(missing_by_snp.median()*100, 2), "%")
print("Median individual missingness:", round(missing_by_individual.median()*100, 2), "%")

display(missing_by_snp.head(10).to_frame("missing_fraction"))


In [ ]:
# Select a target SNP with high call rate and at least two genotype classes.

call_rate = G.notna().mean()
eligible = []
for col in G.columns:
    observed = G[col].dropna()
    if len(observed) >= int(0.90 * len(G)) and observed.nunique() >= 2:
        eligible.append(col)

if not eligible:
    raise ValueError("No suitable target SNP found. Lower the call-rate threshold or inspect the genotype matrix.")

target_snp = eligible[0]
print("Selected target SNP:", target_snp)
print("Call rate:", round(call_rate[target_snp]*100, 2), "%")
print("Genotype distribution:")
display(G[target_snp].value_counts(dropna=False).sort_index().to_frame("count"))


In [ ]:
# Choose up to 50 predictor SNPs with the highest genotype variance.
# The target SNP is excluded from the predictors.

predictor_candidates = [c for c in eligible if c != target_snp]
variances = G[predictor_candidates].var(skipna=True).sort_values(ascending=False)
feature_cols = variances.head(min(50, len(variances))).index.tolist()

model_df = G[feature_cols + [target_snp]].dropna()
print("Complete cases available for baseline model:", model_df.shape[0])
print("Number of predictor SNPs:", len(feature_cols))


## Assignment answers at a glance

**Problem:** Missing or uncertain genotypes in sequencing-derived DArTseq SNP datasets can reduce the reliability of downstream genomic analyses.

**Intended AI/ML outcome:** Build a machine-learning model that learns genomic patterns from observed SNP genotypes and predicts a missing/hidden genotype accurately.

**Potential inputs (features):** Observed genotype calls at other SNP markers for the same individuals; in a future version, SNP position, call rate, sequencing/read-depth and population-structure variables can also be included.

**Output (target):** The genotype class at a selected SNP, represented as 0, 1, or 2 after verifying the actual coding in the DArTseq file.

**ML workflow:** Real DArTseq data → inspect/clean → identify genotype coding → assess missingness → select features and target SNP → mask known target genotypes → train ML model → predict masked genotypes → evaluate prediction accuracy.


## 6. Proposed ML workflow

```text
Real DArTseq SNP matrix
        |
        v
Data inspection + missingness assessment
        |
        v
Select high-quality / informative SNPs
        |
        v
Choose a target SNP
        |
        v
Mask known target genotypes for testing
        |
        v
Train/test split
        |
        v
Random Forest classifier
        |
        v
Predict masked genotype classes
        |
        v
Accuracy + precision + recall + F1 + confusion matrix
        |
        v
Interpret whether genomic patterns can support imputation
```

**Why this workflow?** It converts the genomics problem into a supervised classification task. The model learns from observed genotype patterns and predicts the genotype class at the target SNP. The known original target values provide the ground truth for evaluation.
